# Pseudo-Label Generation — Pipeline B

Generate binary puncta pseudo-labels filtered by neurite proximity.
Detects puncta using LoG, detects neurites using the Meijering filter,
and keeps only puncta within a dilated neurite region.

**Strategy:** Detect puncta near dendrites and axons only. This is more
biologically specific than Pipeline A — synaptic puncta on neurites are
the primary indicator of neuroplasticity (Ly et al. 2018, Savage et al. 2024).

**Notebook structure:**
1. Configuration
2. Imports & setup
3. Helper functions (LoG, Meijering, proximity filtering)
4. Visual validation on sample images
5. Batch generation for all images
6. Dataset statistics & summary

## 1. Configuration

All parameters live here. Change these cells, nothing else.

In [ ]:
from types import SimpleNamespace

In [ ]:
cfg = SimpleNamespace(
    seed=42,
)

In [ ]:
data_cfg = SimpleNamespace(
    patch_root="../../data/patches_128",          # directory with .npy patches + index.csv
    output_root="../../data/pseudolabels_B_128",  # output directory for pseudo-label masks
    exclude_patterns=["KONTROLA"],
)

In [ ]:
detect_cfg = SimpleNamespace(
    # --- LoG blob detection (same as Pipeline A) ---
    # Sigma calibrated to puncta diameter 2-5 px at 107 nm/px.
    # blob radius ≈ sqrt(2) * sigma → sigma = (diameter/2) / sqrt(2)
    min_sigma=0.7,
    max_sigma=1.8,
    num_sigma=5,
    log_threshold=0.01,
    overlap=0.5,
    exclude_border=5,
    # Channels for puncta detection: [0, 1] = presynaptic + postsynaptic.
    # Channel 2 (third marker) is excluded from puncta detection unless
    # you know it labels synaptic structures.
    puncta_channels=[0, 1],

    # --- Neurite detection (Meijering filter) ---
    # Meijering (2004) neuriteness filter, designed for neurite tracing.
    # Sigma range matches neurite widths: 5-19 px → radii 2.5-9.5 px.
    neurite_sigmas=list(range(2, 11)),  # sigma 2 through 10
    # Which channel best shows neurite morphology. Set to the channel
    # that labels dendrites/axons most clearly. If unsure, try each
    # channel in the validation section below and pick the best one.
    # Set to None to use max across all channels.
    neurite_channel=None,
    # Neurite mask thresholding method: "otsu" or "percentile".
    neurite_threshold_method="otsu",
    # Only used when neurite_threshold_method="percentile".
    # Top X% of non-zero Meijering response is considered neurite.
    neurite_percentile=90,

    # --- Proximity filtering ---
    # Dilation radius for the near-neurite region, in pixels.
    # Eroglu lab convention (Stogsdill et al. 2017, Ippolito & Eroglu 2010)
    # uses 2-3 px centroid proximity for colocalization (~0.2-0.3 µm).
    # At 107 nm/px, 4 px ≈ 0.43 µm — conservative for capturing boutons
    # slightly off the neurite shaft.
    dilation_radius=4,

    # --- Colocalization (pre+post marker overlap) ---
    # If True, require spatial overlap between presynaptic (ch0) and
    # postsynaptic (ch1) puncta masks. This produces the most biologically
    # meaningful labels but also the sparsest.
    require_colocalization=False,
    # Dilation applied to one channel before checking overlap.
    # Accounts for subpixel misalignment and the ~20-30 nm synaptic cleft.
    # 2 px ≈ 0.21 µm, well within the optical resolution limit.
    colocalization_dilation=2,
)

## 2. Imports & Setup

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from skimage.feature import blob_log
from skimage.draw import disk
from skimage.filters import meijering, threshold_otsu
from skimage.morphology import binary_dilation, disk as morph_disk
from tqdm.auto import tqdm

sys.path.insert(0, os.path.abspath("../.."))
from data_utils.reassemble_patches import (
    reassemble_image,
    slice_to_patches,
    list_image_indices,
)

np.random.seed(cfg.seed)

In [ ]:
output_dir = Path(data_cfg.output_root)
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")

image_indices = list_image_indices(
    data_cfg.patch_root,
    exclude_patterns=data_cfg.exclude_patterns,
)
print(f"Images to process: {len(image_indices)}")

## 3. Helper Functions

In [ ]:
def detect_puncta_log(channel_image, cfg):
    """Detect puncta in a single 2D channel using Laplacian of Gaussian.

    Args:
        channel_image: (H, W) float32 in [0, 1]
        cfg: SimpleNamespace with LoG parameters

    Returns:
        blobs: (N, 3) array of (row, col, sigma)
    """
    blobs = blob_log(
        channel_image,
        min_sigma=cfg.min_sigma,
        max_sigma=cfg.max_sigma,
        num_sigma=cfg.num_sigma,
        threshold=cfg.log_threshold,
        overlap=cfg.overlap,
        exclude_border=cfg.exclude_border,
    )
    return blobs

In [ ]:
def blobs_to_mask(blobs, image_shape):
    """Convert LoG blob detections to a binary mask.

    Each blob is rendered as a filled disk with radius = sqrt(2) * sigma,
    following the scikit-image convention for LoG blob radius.
    """
    mask = np.zeros(image_shape, dtype=np.uint8)
    for row, col, sigma in blobs:
        radius = max(1, int(np.round(np.sqrt(2) * sigma)))
        rr, cc = disk((int(row), int(col)), radius, shape=image_shape)
        mask[rr, cc] = 1
    return mask

In [ ]:
def detect_neurites(image, cfg):
    """Detect neurite structures using the Meijering neuriteness filter.

    The Meijering filter (2004) was designed specifically for neurite tracing.
    It enhances ridge-like structures at multiple scales by analyzing
    eigenvalues of the Hessian matrix.

    Args:
        image: (C, H, W) float32 in [0, 1]
        cfg: SimpleNamespace with neurite detection parameters

    Returns:
        neuriteness: (H, W) float32 response map
        neurite_mask: (H, W) bool binary mask of detected neurites
    """
    # Select the channel for neurite detection
    if cfg.neurite_channel is not None:
        input_img = image[cfg.neurite_channel]
    else:
        # Use max across channels if no specific channel is set
        input_img = np.max(image, axis=0)

    neuriteness = meijering(
        input_img,
        sigmas=cfg.neurite_sigmas,
        black_ridges=False,  # bright neurites on dark background (fluorescence)
    )

    # Threshold into binary mask
    # Only threshold non-zero values (background response is near zero)
    nonzero = neuriteness[neuriteness > 0]
    if len(nonzero) == 0:
        return neuriteness, np.zeros_like(neuriteness, dtype=bool)

    if cfg.neurite_threshold_method == "otsu":
        t = threshold_otsu(nonzero)
    elif cfg.neurite_threshold_method == "percentile":
        t = np.percentile(nonzero, cfg.neurite_percentile)
    else:
        raise ValueError(f"Unknown threshold method: {cfg.neurite_threshold_method}")

    neurite_mask = neuriteness > t
    return neuriteness, neurite_mask

In [ ]:
def generate_pseudolabels_B(image, detect_cfg):
    """Generate Pipeline-B pseudo-labels for one full image.

    Steps:
    1. Detect puncta per channel using LoG
    2. Detect neurites using Meijering filter
    3. Dilate neurite mask to create proximity region
    4. Optionally require pre+post colocalization
    5. Filter: keep only puncta within the near-neurite region

    Args:
        image: (C, H, W) float32 in [0, 1]
        detect_cfg: SimpleNamespace with all detection parameters

    Returns:
        label_mask: (H, W) uint8 binary mask
        intermediates: dict with neurite_mask, puncta_mask, near_neurite, etc.
        stats: dict with counts
    """
    C, H, W = image.shape

    # Step 1: Detect puncta per channel
    per_channel_blobs = {}
    per_channel_masks = {}
    for ch in detect_cfg.puncta_channels:
        blobs = detect_puncta_log(image[ch], detect_cfg)
        per_channel_blobs[ch] = blobs
        per_channel_masks[ch] = blobs_to_mask(blobs, (H, W)) if len(blobs) > 0 else np.zeros((H, W), dtype=np.uint8)

    # Step 2: Combine puncta masks
    if detect_cfg.require_colocalization and len(detect_cfg.puncta_channels) >= 2:
        ch_pre = detect_cfg.puncta_channels[0]
        ch_post = detect_cfg.puncta_channels[1]
        # Dilate presynaptic mask to allow subpixel misalignment
        pre_dilated = binary_dilation(
            per_channel_masks[ch_pre].astype(bool),
            morph_disk(detect_cfg.colocalization_dilation),
        )
        puncta_mask = (pre_dilated & per_channel_masks[ch_post].astype(bool)).astype(np.uint8)
    else:
        # Union of all puncta channels
        puncta_mask = np.zeros((H, W), dtype=np.uint8)
        for ch in detect_cfg.puncta_channels:
            puncta_mask = np.maximum(puncta_mask, per_channel_masks[ch])

    # Step 3: Detect neurites
    neuriteness, neurite_mask = detect_neurites(image, detect_cfg)

    # Step 4: Dilate neurite mask for proximity region
    near_neurite = binary_dilation(
        neurite_mask,
        morph_disk(detect_cfg.dilation_radius),
    )

    # Step 5: Filter puncta by proximity to neurites
    label_mask = (puncta_mask.astype(bool) & near_neurite).astype(np.uint8)

    # Collect intermediates for visualization
    intermediates = {
        "neuriteness": neuriteness,
        "neurite_mask": neurite_mask,
        "near_neurite": near_neurite,
        "puncta_mask_raw": puncta_mask,
        "per_channel_blobs": per_channel_blobs,
    }

    n_blobs_raw = sum(len(b) for b in per_channel_blobs.values())
    stats = {
        "total_blobs_raw": n_blobs_raw,
        "per_channel": {ch: len(b) for ch, b in per_channel_blobs.items()},
        "puncta_px_raw": int(puncta_mask.sum()),
        "neurite_fraction": float(neurite_mask.mean()),
        "near_neurite_fraction": float(near_neurite.mean()),
        "label_fraction": float(label_mask.mean()),
        "puncta_retained_fraction": float(label_mask.sum()) / max(1, puncta_mask.sum()),
    }
    return label_mask, intermediates, stats

## 4. Visual Validation

Inspect detection results on a sample image before running the full batch.
This section is critical: verify that the Meijering filter captures actual
neurites and that puncta are correctly filtered.

In [ ]:
sample_idx = image_indices[0]
full_img, records = reassemble_image(data_cfg.patch_root, sample_idx)
print(f"Image {sample_idx}: shape={full_img.shape}, "
      f"range=[{full_img.min():.3f}, {full_img.max():.3f}]")

In [ ]:
label_mask, intermediates, stats = generate_pseudolabels_B(full_img, detect_cfg)
print(f"Detection stats:")
for k, v in stats.items():
    print(f"  {k}: {v}")

### Channel comparison for neurite detection

Run the Meijering filter on each channel independently to determine which
channel best shows neurite morphology. Set `detect_cfg.neurite_channel`
accordingly.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
ch_names = ["Ch0 (presynaptic)", "Ch1 (postsynaptic)", "Ch2 (third marker)"]

for c in range(min(3, full_img.shape[0])):
    # Raw channel
    axes[0, c].imshow(full_img[c], cmap="gray", vmin=0, vmax=0.5)
    axes[0, c].set_title(ch_names[c])
    axes[0, c].axis("off")

    # Meijering response
    resp = meijering(full_img[c], sigmas=detect_cfg.neurite_sigmas, black_ridges=False)
    axes[1, c].imshow(resp, cmap="hot")
    axes[1, c].set_title(f"Meijering response (Ch{c})")
    axes[1, c].axis("off")

plt.suptitle("Choose the channel where neurites appear most clearly in the Meijering response.\n"
             "Then set detect_cfg.neurite_channel = <best channel index>.",
             fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Full pipeline overview
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

composite = np.max(full_img, axis=0)

# Row 1: inputs
axes[0, 0].imshow(composite, cmap="gray", vmin=0, vmax=0.5)
axes[0, 0].set_title("Max-channel composite")

axes[0, 1].imshow(intermediates["neuriteness"], cmap="hot")
axes[0, 1].set_title("Meijering neuriteness")

axes[0, 2].imshow(intermediates["neurite_mask"], cmap="gray")
axes[0, 2].set_title(f"Neurite mask ({stats['neurite_fraction']:.2%} of image)")

# Row 2: outputs
axes[1, 0].imshow(intermediates["near_neurite"], cmap="gray")
axes[1, 0].set_title(f"Near-neurite zone (dilation={detect_cfg.dilation_radius}px)")

axes[1, 1].imshow(composite, cmap="gray", vmin=0, vmax=0.5)
axes[1, 1].imshow(intermediates["puncta_mask_raw"], cmap="Reds",
                  alpha=0.4 * intermediates["puncta_mask_raw"])
axes[1, 1].set_title(f"All detected puncta (raw)")

axes[1, 2].imshow(composite, cmap="gray", vmin=0, vmax=0.5)
axes[1, 2].imshow(label_mask, cmap="Reds", alpha=0.4 * label_mask)
axes[1, 2].set_title(f"Final pseudo-labels (near-neurite only, "
                     f"{stats['puncta_retained_fraction']:.0%} retained)")

for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Zoomed-in crop for detailed inspection
crop_y, crop_x = 400, 400
crop_size = 256
sy = slice(crop_y, crop_y + crop_size)
sx = slice(crop_x, crop_x + crop_size)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Row 1: raw channels with blob circles
for c in range(min(3, full_img.shape[0])):
    axes[0, c].imshow(full_img[c, sy, sx], cmap="gray", vmin=0, vmax=0.5)
    if c in intermediates["per_channel_blobs"]:
        for row, col, sigma in intermediates["per_channel_blobs"][c]:
            if crop_y <= row < crop_y + crop_size and crop_x <= col < crop_x + crop_size:
                r = np.sqrt(2) * sigma
                circ = Circle((col - crop_x, row - crop_y), r,
                              fill=False, edgecolor="red", linewidth=0.8)
                axes[0, c].add_patch(circ)
    axes[0, c].set_title(ch_names[c] if c < len(ch_names) else f"Ch{c}")
    axes[0, c].axis("off")

# Row 2: neurite mask, raw puncta, filtered puncta
axes[1, 0].imshow(intermediates["near_neurite"][sy, sx], cmap="gray")
axes[1, 0].set_title("Near-neurite zone")
axes[1, 0].axis("off")

axes[1, 1].imshow(intermediates["puncta_mask_raw"][sy, sx], cmap="gray")
axes[1, 1].set_title("Puncta (all)")
axes[1, 1].axis("off")

axes[1, 2].imshow(composite[sy, sx], cmap="gray", vmin=0, vmax=0.5)
axes[1, 2].imshow(label_mask[sy, sx], cmap="Reds",
                  alpha=0.5 * label_mask[sy, sx])
axes[1, 2].set_title("Final pseudo-labels")
axes[1, 2].axis("off")

plt.suptitle(f"Crop ({crop_y}:{crop_y+crop_size}, {crop_x}:{crop_x+crop_size})")
plt.tight_layout()
plt.show()

In [ ]:
# Sanity check: slice back to patches and visualize a few
patch_labels = slice_to_patches(label_mask, records)
print(f"Generated {len(patch_labels)} patch labels")

# Pick patches that have some labels (not all-zero)
nonempty = [(f, l) for f, l in patch_labels.items() if l.sum() > 0]
print(f"Patches with at least one labeled pixel: {len(nonempty)}/{len(patch_labels)}")

sample_fnames = [f for f, _ in nonempty[:8]]
if len(sample_fnames) < 8:
    # Pad with some empty ones for comparison
    empty_fnames = [f for f in patch_labels if f not in [s for s, _ in nonempty]][:8-len(sample_fnames)]
    sample_fnames.extend(empty_fnames)

fig, axes = plt.subplots(2, len(sample_fnames), figsize=(2.5 * len(sample_fnames), 5))
if len(sample_fnames) == 1:
    axes = axes.reshape(2, 1)
for i, fname in enumerate(sample_fnames):
    patch_img = np.load(Path(data_cfg.patch_root) / fname)
    patch_lbl = patch_labels[fname]
    axes[0, i].imshow(np.max(patch_img, axis=0), cmap="gray", vmin=0, vmax=0.5)
    axes[0, i].set_title(f"{fname[:12]}...", fontsize=7)
    axes[0, i].axis("off")
    axes[1, i].imshow(patch_lbl, cmap="gray")
    n_pos = int(patch_lbl.sum())
    axes[1, i].set_title(f"{n_pos} px", fontsize=7)
    axes[1, i].axis("off")
plt.suptitle("Top: image patches | Bottom: pseudo-label masks (Pipeline B)")
plt.tight_layout()
plt.show()

## 5. Batch Generation

Process all images: reassemble → detect puncta → detect neurites → filter → slice → save.
Each pseudo-label is saved as a `(1, 128, 128)` float32 `.npy` file.

In [ ]:
import csv

all_stats = []

for img_idx in tqdm(image_indices, desc="Generating pseudo-labels (Pipeline B)"):
    try:
        full_img, records = reassemble_image(
            data_cfg.patch_root, img_idx,
            exclude_patterns=data_cfg.exclude_patterns,
        )
    except ValueError as e:
        print(f"  Skipping image {img_idx}: {e}")
        continue

    label_mask, intermediates, stats = generate_pseudolabels_B(full_img, detect_cfg)
    stats["image_index"] = img_idx
    stats["source_image"] = records[0]["source_image"]

    # Slice into patches and save
    patch_labels = slice_to_patches(label_mask, records)
    for fname, patch_lbl in patch_labels.items():
        out = patch_lbl.astype(np.float32)[np.newaxis, ...]  # (1, 128, 128)
        np.save(output_dir / fname, out)

    stats["n_patches"] = len(patch_labels)
    stats["n_patches_nonempty"] = sum(1 for p in patch_labels.values() if p.sum() > 0)
    all_stats.append(stats)

print(f"\nDone. Processed {len(all_stats)} images.")
print(f"Total patches saved: {sum(s['n_patches'] for s in all_stats)}")
print(f"Non-empty patches: {sum(s['n_patches_nonempty'] for s in all_stats)}")

In [ ]:
import shutil
src_csv = Path(data_cfg.patch_root) / "index.csv"
dst_csv = output_dir / "index.csv"
shutil.copy2(src_csv, dst_csv)
print(f"Copied index.csv to {dst_csv}")

## 6. Dataset Statistics

In [ ]:
total_blobs = [s["total_blobs_raw"] for s in all_stats]
label_fracs = [s["label_fraction"] for s in all_stats]
neurite_fracs = [s["neurite_fraction"] for s in all_stats]
retained = [s["puncta_retained_fraction"] for s in all_stats]

print(f"Puncta per image (raw):   min={min(total_blobs)}, "
      f"median={int(np.median(total_blobs))}, max={max(total_blobs)}")
print(f"Neurite area fraction:    min={min(neurite_fracs):.4f}, "
      f"median={np.median(neurite_fracs):.4f}, max={max(neurite_fracs):.4f}")
print(f"Puncta retained fraction: min={min(retained):.2f}, "
      f"median={np.median(retained):.2f}, max={max(retained):.2f}")
print(f"Label pixel fraction:     min={min(label_fracs):.5f}, "
      f"median={np.median(label_fracs):.5f}, max={max(label_fracs):.5f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(total_blobs, bins=20, edgecolor="black", alpha=0.7)
axes[0, 0].set_xlabel("Raw puncta per image")
axes[0, 0].set_title("Raw puncta count (before neurite filtering)")

axes[0, 1].hist(neurite_fracs, bins=20, edgecolor="black", alpha=0.7, color="green")
axes[0, 1].set_xlabel("Fraction of image")
axes[0, 1].set_title("Neurite area fraction")

axes[1, 0].hist(retained, bins=20, edgecolor="black", alpha=0.7, color="orange")
axes[1, 0].set_xlabel("Fraction retained")
axes[1, 0].set_title("Puncta retained after neurite filtering")

axes[1, 1].hist(label_fracs, bins=20, edgecolor="black", alpha=0.7, color="red")
axes[1, 1].set_xlabel("Fraction of pixels labeled")
axes[1, 1].set_title("Final label sparsity")

for ax in axes.flat:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
stats_path = output_dir / "detection_stats.csv"
fieldnames = ["image_index", "source_image", "total_blobs_raw",
              "neurite_fraction", "near_neurite_fraction",
              "puncta_retained_fraction", "label_fraction",
              "n_patches", "n_patches_nonempty"]
with open(stats_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(all_stats)
print(f"Stats saved to {stats_path}")

In [ ]:
import json

config_path = output_dir / "detect_config.json"
with open(config_path, "w") as f:
    json.dump({k: v for k, v in vars(detect_cfg).items()}, f, indent=2)
print(f"Config saved to {config_path}")

## Next Steps

1. **Inspect the channel comparison** above. Set `detect_cfg.neurite_channel`
   to the channel that best shows neurites in the Meijering response.
2. **Check `puncta_retained_fraction`**: If below 30%, neurite detection might
   be too strict (increase `dilation_radius` or switch to `percentile` threshold
   with a lower percentile). If above 90%, filtering is not adding much value.
3. **Try colocalization mode**: Set `detect_cfg.require_colocalization = True`
   for the most biologically meaningful labels (only pre+post overlapping puncta).
   Warning: this will produce very sparse labels.
4. **Compare with Pipeline A**: Train SwinUNETR on both label sets and compare
   Dice scores to measure the impact of neurite proximity filtering.
5. **Train segmentation**: Use these pseudo-labels with `finetune_swinunetr_seg.ipynb`:
   ```
   data_cfg.label_root = "../../data/pseudolabels_B_128"
   ```